# 17 Bayesian answers: Bayes factors, credible intervals and Bayesian models

simplydata's `bayesian=True` (added after 0.2.0), checked on

- **Student's sleep data** (Cushny & Peebles 1905): `compare_averages_of` (two groups), `compare_before_and_after`
  (paired) and `compare_average_to` (one column against a number), all JZS Bayes factor t-tests;
- **Fisher's iris** and **mtcars**: `find_correlation_between`;
- **Hraba & Grant's (1970) dolls** and **mtcars**: `compare_counts_of` (Gunel-Dickey Bayes factor, independent
  multinomial sampling with the group totals fixed);
- **mtcars** and **sleepstudy** (Belenky et al. 2003): `explain(..., bayesian=True)`, Bayesian regression from R
  rstanarm (`stan_glm` for numbers and for yes/no, `stan_glmer` with `grouped_by=`).

**Only simplydata runs in this notebook.** The reference values (`references/17_bayesian.csv`, from
`scripts/references/17_bayesian.py`, which runs R BayesFactor, rstanarm and posterior, and the formulas in SciPy)
are described below; every check prints its source. The reference script fits six long rstanarm models, so this
notebook is run on GitHub Actions.

**References and tolerances.**

- **Published values**: the Bayes factors printed in the BayesFactor manual (Morey & Rouder, vignette "Using the
  'BayesFactor' package", version 0.9.12-4.8): sleep, paired, 17.259; iris, Sepal.Length with Sepal.Width, 0.509;
  Hraba & Grant's dolls, 1.81. Half a unit of the last printed digit.
- **R BayesFactor**, run here with the same call (`ttestBF`, `correlationBF`, `contingencyTableBF(sampleType =
  "indepMulti")`): a Bayes factor is worked out without random draws, so simplydata must agree to a relative
  difference of 1e-9.
- **The published formulas in Python** (independent of R): the JZS t-test integral of Rouder et al. (2009,
  equation 1) and the exact Jeffreys correlation Bayes factor with a stretched beta prior (Ly, Verhagen &
  Wagenmakers 2016), both by SciPy `quad`; the Gunel & Dickey (1974) closed form for independent multinomials (as
  in Jamil et al. 2017) with SciPy `gammaln`. BayesFactor computes the t-test and correlation Bayes factors by
  numerical integration and reports its own proportional error (`extractBF()$error`); the tolerance is 4 times that
  reported error (at least 1e-8 relative). The closed form is exact: 1e-9 relative.
- **Credible intervals come from random draws.** Two checks each:
  1. *same draws*: R run here with the same seed (BayesFactor `posterior(iterations = 10000)` after
     `set.seed(1)` with R's default generators; rstanarm `seed = 1`, 4 chains of 2000 iterations) must give the
     same numbers, relative difference 1e-9;
  2. *Monte Carlo*: an independent, much longer run with another seed (BayesFactor 200,000 draws; rstanarm 4
     chains of 10,000 iterations). simplydata's median and 2.5 % and 97.5 % quantiles must lie within
     4 Monte Carlo standard errors of it, `4 * sqrt(MCSE_simplydata^2 + MCSE_long^2)`, the MCSEs from
     `posterior::mcse_median()` and `posterior::mcse_quantile()` (Vehtari et al. 2021) on each run's own draws
     (simplydata's draws are the same-seed draws of check 1). With 4 MCSEs a correct answer fails about
     once in 16,000 comparisons.
- R-hat and effective draws (rstanarm) are compared with the same-seed run only (1e-9 relative).

No tolerance is loosened when a comparison fails; a FAIL is reported and explained in FINDINGS.md.

In [1]:
import simplydata as sdata

from checks import ReferenceChecks, data_file, quiet, read_row, read_rows, versions

NOTEBOOK = "17_bayesian"
versions("simplydata")
checks = ReferenceChecks(NOTEBOOK)


def load(name):
    return quiet(sdata.table, data_file(NOTEBOOK, name), separator=",")


QUANTILES = {"2.5 % quantile": "low", "median": "middle", "97.5 % quantile": "high"}


def check_interval(dataset, method, low, middle, high):
    """simplydata's credible interval and middle against R with the same draws and against a long run."""
    ours = {"low": low, "middle": middle, "high": high}
    checks.check_values(dataset, method, {name: ours[key] for name, key in QUANTILES.items()})


sleep = load("sleep")
mtcars = load("mtcars")

Python 3.12.16 on Linux 7.0.0-1020-azure
simplydata 0.2.0 (commit 3f0192b of github.com/bioforkgithub/simplydata)
simplydata 0.2.0


R version 4.6.2 (2026-10-29)


## The published formulas in Python, and the two checks of a credible interval (in the reference script)

`jzs()` is equation 1 of Rouder et al. (2009) with prior scale r; `jeffreys_correlation()` is the exact Bayes factor
of Ly et al. (2016) for a correlation, h(rho) = (1 - rho^2)^((n-1)/2) 2F1((n-1)/2, (n-1)/2; 1/2; (r rho)^2) (the odd
part of h integrates to 0 against a symmetric prior), with BayesFactor's stretched beta prior of width w on rho:
Beta(1/w, 1/w) on (rho + 1) / 2. `gunel_dickey()` is the independent-multinomial Bayes factor with a = 1, the rows
(groups) fixed.

`interval_references()` (in the reference script) runs R twice: with simplydata's seed and number of draws (the same draws), and a long run with
another seed; it gives the quantiles and their Monte Carlo standard errors.

## Two groups: `compare_averages_of(bayesian=True)` on Student's sleep data

The 10 patients' extra sleep with drug 1 and with drug 2, treated here as two independent groups (as in the
example of BayesFactor's `ttestBF`). The full answer once:

In [2]:
result = sleep.compare_averages_of("extra", for_each_group_in="group", bayesian=True)

Note: "bayesian=True" is now called test_method="bayesian". The old name will stop working in a later version.


Comparing the average "extra" in drug 1 and drug 2 the Bayesian way (10 and 10 rows).
The data are 1.27 times as likely if the averages of drug 1 and drug 2 differ as if they do not differ (Bayes factor for a difference: 1.27, with the JZS prior: a Cauchy prior on the effect size (the difference in standard deviations) with width 0.707 ("medium", BayesFactor's default; Rouder et al. 2009)).
Given the data and the prior, the difference (drug 1 minus drug 2) is probably between -2.89 and 0.349 (95 % credible interval: given the data and the prior, a 95 % chance that the value lies in it); its middle value (the posterior median) is -1.15. The effect size (the difference in standard deviations) is probably between -1.50 and 0.163.
Method: R BayesFactor::ttestBF(x, y) (rscale = "medium"); the interval from 10,000 draws of the posterior (BayesFactor::posterior()), with_seed=1: the same seed gives the same draws again.
A Bayes factor is not the chance that a hypothesis is true: it says how mu

In [3]:
for width in ("medium", "wide", "ultrawide"):
    found = read_row(quiet(sleep.compare_averages_of, "extra", for_each_group_in="group", bayesian=True,
                           prior_width=width))
    dataset, method = "Student's sleep data", f"compare_averages_of (bayesian, prior {width})"
    checks.check(dataset, method, "Bayes factor", found["Bayes factor"])
    if width == "medium":
        check_interval(dataset, method + ": difference", found["difference 95 % credible interval low"],
                       found["difference"], found["difference 95 % credible interval high"])

agrees  Student's sleep data | compare_averages_of (bayesian, prior medium) | Bayes factor: simplydata 1.26592514958, R BayesFactor ttestBF 1.26592514958; relative difference 0.00e+00 (allowed 1e-09)
      Source: R BayesFactor ttestBF (run reference) - https://search.r-project.org/CRAN/refmans/BayesFactor/html/ttestBF.html (version 0.9.12.4.8)
agrees  Student's sleep data | compare_averages_of (bayesian, prior medium) | Bayes factor: simplydata 1.26592514958, SciPy quad of the JZS integral (Rouder et al. 2009, eq. 1) 1.26592514965; relative difference 5.48e-11 (allowed 1e-04)
      Source: SciPy quad of the JZS integral (Rouder et al. 2009, eq. 1) (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/stats.html
agrees  Student's sleep data | compare_averages_of (bayesian, prior medium): difference | 2.5 % quantile: simplydata -2.88509154479, R BayesFactor posterior (same seed) -2.88509154479; relative difference 0.00e+00 (allowed 1e-09)
      Source: R BayesFactor poster

## Before and after: `compare_before_and_after(bayesian=True)` on Student's sleep data

The same 10 patients on both drugs, as Student (1908) analysed them. The BayesFactor manual prints the Bayes factor
17.259 for these differences (`ttestBF(x = diffScores)`, prior width "medium").

In [4]:
found = read_row(quiet(load("sleep_wide").compare_before_and_after, "drug_1", "drug_2", bayesian=True))
dataset, method = "Student's sleep data", "compare_before_and_after (bayesian)"
checks.check(dataset, method, "Bayes factor", found["Bayes factor"])
check_interval(dataset, method + ": average change", found["average change 95 % credible interval low"],
               found["average change"], found["average change 95 % credible interval high"])

agrees  Student's sleep data | compare_before_and_after (bayesian) | Bayes factor: simplydata 17.2588802678, BayesFactor manual (0.9.12-4.8), printed 17.259; difference 1.20e-04 (allowed 5e-04)
      Source: BayesFactor manual (0.9.12-4.8), printed (printed) - https://cran.r-project.org/web/packages/BayesFactor/vignettes/manual.html
agrees  Student's sleep data | compare_before_and_after (bayesian) | Bayes factor: simplydata 17.2588802678, R BayesFactor ttestBF 17.2588802678; relative difference 0.00e+00 (allowed 1e-09)
      Source: R BayesFactor ttestBF (run reference) - https://search.r-project.org/CRAN/refmans/BayesFactor/html/ttestBF.html (version 0.9.12.4.8)
agrees  Student's sleep data | compare_before_and_after (bayesian) | Bayes factor: simplydata 17.2588802678, SciPy quad of the JZS integral (Rouder et al. 2009, eq. 1) 17.2588802678; relative difference 8.30e-14 (allowed 2e-07)
      Source: SciPy quad of the JZS integral (Rouder et al. 2009, eq. 1) (run reference) - https://

## One column against a number: `compare_average_to(bayesian=True)`

Drug 1's extra sleep against 1 hour, prior width "wide" (r = 1). The Bayes factor is R's `ttestBF(x, mu = 1)`;
BayesFactor's `posterior()` leaves `mu =` out, so the interval of the average minus 1 comes from
`ttestBF(x - 1)` (the same Bayes factor).

In [5]:
found = read_row(quiet(load("drug_1").compare_average_to, "extra", 1, bayesian=True, prior_width="wide"))
dataset, method = "Student's sleep data (drug 1)", "compare_average_to 1 (bayesian, prior wide)"
checks.check(dataset, method, "Bayes factor", found["Bayes factor"])
check_interval(dataset, method + ": difference", found["difference 95 % credible interval low"], found["difference"],
               found["difference 95 % credible interval high"])

agrees  Student's sleep data (drug 1) | compare_average_to 1 (bayesian, prior wide) | Bayes factor: simplydata 0.254785902548, R BayesFactor ttestBF 0.254785902548; relative difference 2.18e-16 (allowed 1e-09)
      Source: R BayesFactor ttestBF (run reference) - https://search.r-project.org/CRAN/refmans/BayesFactor/html/ttestBF.html (version 0.9.12.4.8)
agrees  Student's sleep data (drug 1) | compare_average_to 1 (bayesian, prior wide) | Bayes factor: simplydata 0.254785902548, SciPy quad of the JZS integral (Rouder et al. 2009, eq. 1) 0.254785902553; relative difference 2.22e-11 (allowed 8e-04)
      Source: SciPy quad of the JZS integral (Rouder et al. 2009, eq. 1) (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/stats.html
agrees  Student's sleep data (drug 1) | compare_average_to 1 (bayesian, prior wide): difference | 2.5 % quantile: simplydata -1.36155600575, R BayesFactor posterior (same seed) -1.36155600575; relative difference 0.00e+00 (allowed 1e-09)
      

## Correlation: `find_correlation_between(bayesian=True)`

Fisher's iris, Sepal.Length with Sepal.Width (all 150 flowers; the BayesFactor manual prints 0.509 for
`correlationBF(y = Sepal.Length, x = Sepal.Width)`), and mtcars, fuel use with weight (a strong correlation).

In [6]:
iris = quiet(sdata.table, "../data/iris.csv", separator=",")
for dataset, data, y, x in [("Fisher's iris", iris, "Sepal.Length", "Sepal.Width"), ("mtcars", mtcars, "mpg", "wt")]:
    found = read_row(quiet(data.find_correlation_between, y, x, bayesian=True))
    method = f"find_correlation_between {y}, {x} (bayesian)"
    checks.check(dataset, method, "Bayes factor", found["Bayes factor"])
    check_interval(dataset, method + ": r", found["r 95 % credible interval low"], found["r"],
                   found["r 95 % credible interval high"])

agrees  Fisher's iris | find_correlation_between Sepal.Length, Sepal.Width (bayesian) | Bayes factor: simplydata 0.509017511648, BayesFactor manual (0.9.12-4.8), printed 0.509; difference 1.75e-05 (allowed 5e-04)
      Source: BayesFactor manual (0.9.12-4.8), printed (printed) - https://cran.r-project.org/web/packages/BayesFactor/vignettes/manual.html
agrees  Fisher's iris | find_correlation_between Sepal.Length, Sepal.Width (bayesian) | Bayes factor: simplydata 0.509017511648, R BayesFactor correlationBF 0.509017511648; relative difference 2.18e-16 (allowed 1e-09)
      Source: R BayesFactor correlationBF (run reference) - https://search.r-project.org/CRAN/refmans/BayesFactor/html/correlationBF.html (version 0.9.12.4.8)
agrees  Fisher's iris | find_correlation_between Sepal.Length, Sepal.Width (bayesian) | Bayes factor: simplydata 0.509017511648, SciPy quad of the exact Jeffreys Bayes factor (Ly et al. 2016) 0.509017511648; relative difference 1.94e-14 (allowed 1e-08)
      Source: Sc

agrees  mtcars | find_correlation_between mpg, wt (bayesian) | Bayes factor: simplydata 13286626.3086, R BayesFactor correlationBF 13286626.3086; relative difference 0.00e+00 (allowed 1e-09)
      Source: R BayesFactor correlationBF (run reference) - https://search.r-project.org/CRAN/refmans/BayesFactor/html/correlationBF.html (version 0.9.12.4.8)
agrees  mtcars | find_correlation_between mpg, wt (bayesian) | Bayes factor: simplydata 13286626.3086, SciPy quad of the exact Jeffreys Bayes factor (Ly et al. 2016) 13286626.3086; relative difference 5.51e-14 (allowed 1e-08)
      Source: SciPy quad of the exact Jeffreys Bayes factor (Ly et al. 2016) (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/stats.html
agrees  mtcars | find_correlation_between mpg, wt (bayesian): r | 2.5 % quantile: simplydata -0.903442066642, R BayesFactor posterior (same seed) -0.903442066642; relative difference 1.23e-16 (allowed 1e-09)
      Source: R BayesFactor posterior (same seed) (run refer

## Counts: `compare_counts_of(bayesian=True)`

Hraba & Grant (1970): 71 white and 89 black children chose a doll; 50 and 48 of them chose the doll of their own
race. The groups (the children) are sampled as independent multinomials with their totals fixed; the BayesFactor
manual prints 1.81 (`contingencyTableBF(raceDolls, sampleType = "indepMulti", fixedMargin = "cols")`, the children
being its columns). mtcars: gearbox (automatic or manual) by engine shape. The shares are the posterior of each
group's share (BayesFactor's `omega[group, value]`).

In [7]:
dolls = quiet(sdata.table, "../data/fetched/race_dolls.csv", separator=",")
for dataset, data, value, group in [("Hraba & Grant dolls", dolls, "doll", "child"), ("mtcars", mtcars, "gearbox", "engine")]:
    found = read_row(quiet(data.compare_counts_of, value, for_each_group_in=group, bayesian=True))
    method = f"compare_counts_of {value} by {group} (bayesian)"
    checks.check(dataset, method, "Bayes factor", found["Bayes factor"])
    shares = [name for name in found if name.startswith("share ") and "credible interval" not in name]
    for name in shares:
        check_interval(dataset, f"{method}: {name}", found[f"{name} 95 % credible interval low"], found[name],
                       found[f"{name} 95 % credible interval high"])

agrees  Hraba & Grant dolls | compare_counts_of doll by child (bayesian) | Bayes factor: simplydata 1.81485573815, BayesFactor manual (0.9.12-4.8), printed 1.81; difference 4.86e-03 (allowed 5e-03)
      Source: BayesFactor manual (0.9.12-4.8), printed (printed) - https://cran.r-project.org/web/packages/BayesFactor/vignettes/manual.html
agrees  Hraba & Grant dolls | compare_counts_of doll by child (bayesian) | Bayes factor: simplydata 1.81485573815, Gunel & Dickey (1974) closed form (SciPy gammaln) 1.81485573815; relative difference 2.79e-14 (allowed 1e-09)
      Source: Gunel & Dickey (1974) closed form (SciPy gammaln) (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.gammaln.html
agrees  Hraba & Grant dolls | compare_counts_of doll by child (bayesian) | Bayes factor: simplydata 1.81485573815, R BayesFactor contingencyTableBF 1.81485573815; relative difference 0.00e+00 (allowed 1e-09)
      Source: R BayesFactor contingencyTableBF (run reference

agrees  mtcars | compare_counts_of gearbox by engine (bayesian) | Bayes factor: simplydata 0.631316369587, Gunel & Dickey (1974) closed form (SciPy gammaln) 0.631316369587; relative difference 3.87e-15 (allowed 1e-09)
      Source: Gunel & Dickey (1974) closed form (SciPy gammaln) (run reference) - https://docs.scipy.org/doc/scipy-1.18.1/reference/generated/scipy.stats.gammaln.html
agrees  mtcars | compare_counts_of gearbox by engine (bayesian) | Bayes factor: simplydata 0.631316369587, R BayesFactor contingencyTableBF 0.631316369587; relative difference 0.00e+00 (allowed 1e-09)
      Source: R BayesFactor contingencyTableBF (run reference) - https://search.r-project.org/CRAN/refmans/BayesFactor/html/contingencyTableBF.html (version 0.9.12.4.8)
agrees  mtcars | compare_counts_of gearbox by engine (bayesian): share automatic in V-shaped | 2.5 % quantile: simplydata 0.436221656508, R BayesFactor posterior (same seed) 0.436221656508; relative difference 2.55e-16 (allowed 1e-09)
      Sour

## Bayesian models: `explain(..., bayesian=True)` (R rstanarm)

Three models: mtcars fuel use from weight and horsepower (`stan_glm`, normal), mtcars gearbox (manual = 1) from
weight (`stan_glm`, binomial; simplydata shows odds ratios, the exponent of the posterior median and interval
ends, which commute with exp), and sleepstudy reaction time over days with a random intercept for each subject
(`stan_glmer`). Each effect is the posterior median with its central 95 % interval. R runs the same model with the
same seed (the same draws: 1e-9), and a long run with 4 chains of 10,000 iterations and seed 20261121 (within 4
combined Monte Carlo standard errors). R-hat and the bulk and tail effective draws are compared with the same-seed
run. The full answer once:

In [8]:
mtcars.explain("mpg", using=["wt", "hp"], bayesian=True)

Explaining "mpg" from "wt" and "hp" the Bayesian way (32 rows). Each effect below holds the other columns fixed:
  intercept: 37.2 (33.8 to 40.5), the expected "mpg" when every number column is 0 and every group column is at its baseline.
  wt: each 1 more "wt" goes with -3.86 more "mpg" (-5.18 to -2.55).
  hp: each 1 more "hp" goes with -0.0318 more "mpg" (-0.0512 to -0.0126).
  spread around the model (sigma): 2.67 (2.11 to 3.55) in "mpg".
Each number: the posterior median with its 95 % credible interval: given the data and the prior, a 95 % chance that the value lies in it. There are no p-values in a Bayesian model.
Prior used (rstanarm's default weakly informative priors, scaled to the spread of the data; R rstanarm::prior_summary()):
    Intercept (after predictors centered)
      Specified prior:
        ~ normal(location = 20, scale = 2.5)
      Adjusted prior:
        ~ normal(location = 20, scale = 15)
    Coefficients
      Specified prior:
        ~ normal(location = [0,0], 

,term,effect,95 % credible interval low,95 % credible interval high,R-hat,effective draws (middle),effective draws (tails)
1,(intercept),37.188185,33.786452,40.499010,1.001460,3733.925763,2842.179625
2,wt,-3.859517,-5.178107,-2.549375,1.001428,2195.781574,2332.010228
3,hp,-0.031783,-0.051238,-0.012580,1.000925,2106.958355,2392.396188
4,spread around the model (sigma),2.668231,2.108432,3.545112,1.000310,2496.186773,2326.539439


In [9]:
sleepstudy = quiet(sdata.table, "../data/fetched/sleepstudy.csv", separator=",")
MODELS = [
    ("mtcars", mtcars, "mpg ~ wt + hp", "mpg", dict(using=["wt", "hp"]), "stan_glm", None),
    ("mtcars", mtcars, "am ~ wt", "am", dict(using=["wt"], outcome="yes or no"), "stan_glm", "odds ratio"),
    ("sleepstudy", sleepstudy, "Reaction ~ Days + (1 | Subject)", "Reaction", dict(using=["Days"], grouped_by="Subject"),
     "stan_glmer", None),
]
for dataset, data, formula, outcome, options, fit, ratio in MODELS:
    method = f"explain {formula} (bayesian, rstanarm {fit})"
    for row in read_rows(quiet(data.explain, outcome, bayesian=True, **options)):
        term = row["term"]
        ours = [row["95 % credible interval low"], row["effect"], row["95 % credible interval high"]]
        values = {f"{term} {name}" + (f" ({ratio})" if ratio else ""): value
                  for name, value in zip(QUANTILES, ours)}
        values.update({f"{term} {name}": row[name] for name in ("R-hat", "effective draws (middle)",
                                                                 "effective draws (tails)")})
        # the terms R reports: the fixed effects and sigma
        if term in ["(intercept)", *options["using"], "spread around the model (sigma)"]:
            checks.check_values(dataset, method, values)

agrees  mtcars | explain mpg ~ wt + hp (bayesian, rstanarm stan_glm) | (intercept) 2.5 % quantile: simplydata 33.7864523762, R rstanarm (same seed) 33.7864523762; relative difference 0.00e+00 (allowed 1e-09)
      Source: R rstanarm (same seed) (run reference) - https://cran.r-project.org/package=rstanarm (version 2.32.2)
agrees  mtcars | explain mpg ~ wt + hp (bayesian, rstanarm stan_glm) | (intercept) 2.5 % quantile: simplydata 33.7864523762, R rstanarm (4 x 10,000 iterations, other seed) 33.901297779; difference 1.15e-01 (allowed 5e-01)
      Source: R rstanarm (4 x 10,000 iterations, other seed) (run reference) - https://cran.r-project.org/package=rstanarm (version 2.32.2)
agrees  mtcars | explain mpg ~ wt + hp (bayesian, rstanarm stan_glm) | (intercept) median: simplydata 37.1881854462, R rstanarm (same seed) 37.1881854462; relative difference 0.00e+00 (allowed 1e-09)
      Source: R rstanarm (same seed) (run reference) - https://cran.r-project.org/package=rstanarm (version 2.32.2

agrees  mtcars | explain am ~ wt (bayesian, rstanarm stan_glm) | (intercept) 2.5 % quantile (odds ratio): simplydata 122.574404904, R rstanarm (same seed) 122.574404904; relative difference 9.27e-16 (allowed 1e-09)
      Source: R rstanarm (same seed) (run reference) - https://cran.r-project.org/package=rstanarm (version 2.32.2)
agrees  mtcars | explain am ~ wt (bayesian, rstanarm stan_glm) | (intercept) 2.5 % quantile (odds ratio): simplydata 122.574404904, R rstanarm (4 x 10,000 iterations, other seed) 122.267707994; difference 3.07e-01 (allowed 7e+01)
      Source: R rstanarm (4 x 10,000 iterations, other seed) (run reference) - https://cran.r-project.org/package=rstanarm (version 2.32.2)
agrees  mtcars | explain am ~ wt (bayesian, rstanarm stan_glm) | (intercept) median (odds ratio): simplydata 35049.4043374, R rstanarm (same seed) 35049.4043374; relative difference 0.00e+00 (allowed 1e-09)
      Source: R rstanarm (same seed) (run reference) - https://cran.r-project.org/package=rs

agrees  sleepstudy | explain Reaction ~ Days + (1 | Subject) (bayesian, rstanarm stan_glmer) | (intercept) 2.5 % quantile: simplydata 230.507628269, R rstanarm (same seed) 230.507628269; relative difference 1.23e-16 (allowed 1e-09)
      Source: R rstanarm (same seed) (run reference) - https://cran.r-project.org/package=rstanarm (version 2.32.2)
agrees  sleepstudy | explain Reaction ~ Days + (1 | Subject) (bayesian, rstanarm stan_glmer) | (intercept) 2.5 % quantile: simplydata 230.507628269, R rstanarm (4 x 10,000 iterations, other seed) 232.020570594; difference 1.51e+00 (allowed 4e+00)
      Source: R rstanarm (4 x 10,000 iterations, other seed) (run reference) - https://cran.r-project.org/package=rstanarm (version 2.32.2)
agrees  sleepstudy | explain Reaction ~ Days + (1 | Subject) (bayesian, rstanarm stan_glmer) | (intercept) median: simplydata 250.176303392, R rstanarm (same seed) 250.176303392; relative difference 1.14e-16 (allowed 1e-09)
      Source: R rstanarm (same seed) (run

## Result

In [10]:
results = checks.finish()

17_bayesian: 180 comparisons - 179 agrees, 1 explained difference – machine
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)
